# 18 · Buffer study area against shapely

**Question.** Does Fieldwork's `buffer_area` widget expand a boundary by the distance it says?

**This widget cannot agree exactly with a reference, and this check says so.** Turf buffers in
a local azimuthal equidistant projection; the reference buffers the same boundary in UTM at the
same quadrant count. Those are different projections, so demanding equality would demand
something neither side can deliver. The criterion is a **relative area difference** with the
quadrant count declared.

Criteria fixed before running: the area within 0.5% of a UTM reference buffer at the same 8
segments per quadrant, and the result containing the original boundary.

In [ ]:
# Only what this notebook imports. Requesting rasterio alongside pyproj kills this kernel
# outright; see 00-kernel-check for the measured map.
NEEDED = ["pyproj", "shapely", "numpy"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested from the Pyodide distribution:", ", ".join(NEEDED))
except ImportError:
    print("Not a Pyodide kernel; using the ambient environment.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)

## Parity

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_18_buffer_area import run

report = run(Path.cwd() / "fixtures" / "buffer-area-soho-001.json")
print("criterion:", report["criterion"])
print()
print("agrees:", report["agrees"])
print(f"  source boundary          {report['sourceAreaSquareMetres']:12.0f} m2")
print(f"  buffered by {report['distanceMetres']} m        {report['fieldworkAreaSquareMetres']:12.0f} m2")
print(f"  {report['referenceCRS']} reference     {report['referenceAreaSquareMetres']:12.0f} m2")
print(f"  relative difference      {report['relativeAreaDifference']:12.3e}")
print(f"  contains the original    {report['containsTheOriginal']}")

## Why 8 segments and not 32

`buffer_point` uses 32 segments per quadrant; this widget uses 8. The difference is not
inconsistency, it is what each edge is for. A scope's edge **decides membership**, so
`buffer_point` pays for the finer approximation. This boundary is an **acquisition extent** —
it exists so that a network or raster fetched for the study area reaches past its edge — and
its edge decides nothing.

The closed form for a polygon inscribed in a circle says what each setting costs.

In [ ]:
import math

def fraction(vertices):
    return (vertices / (2 * math.pi)) * math.sin(2 * math.pi / vertices)

print(f"{'steps/quadrant':>14} {'vertices':>9} {'area of the circle':>20} {'missing':>10}")
for steps in (1, 2, 4, 8, 16, 32, 64):
    vertices = 4 * steps
    share = fraction(vertices)
    mark = "  <- buffer_area" if steps == 8 else ("  <- buffer_point" if steps == 32 else "")
    print(f"{steps:>14} {vertices:>9} {share:>19.5%} {1 - share:>9.3%}{mark}")

print()
radius = 500
ideal = math.pi * radius ** 2
print(f"On a {radius} m radius, {ideal:.0f} m2 of ideal circle:")
for steps in (8, 32):
    missing = ideal * (1 - fraction(4 * steps))
    print(f"  {steps:>2} steps leaves {missing:7.1f} m2 outside the polygon ({missing / ideal:.3%})")
print()
print("Going from 8 to 32 recovers most of what is missing, and doubling again recovers little.")
print("Both of this project's settings sit either side of the knee, deliberately.")

## Dive deeper · the buffer only matters at the corners

Buffering a rectangle by a distance adds a frame of that width along each side and a quarter
circle at each corner. The approximation touches **only the corners**, so how much it costs
depends on how much of the added area is corner — which depends on the shape being buffered,
not on the distance alone.

In [ ]:
import numpy as np, pyproj, shapely
from shapely.geometry import shape

fixture = json.loads((Path.cwd() / "fixtures" / "buffer-area-soho-001.json").read_text())
source = shape(fixture["input"]["boundary"])
to_utm = pyproj.Transformer.from_crs("OGC:CRS84", report["referenceCRS"], always_xy=True)
project = lambda g: shapely.transform(
    g, lambda c: np.column_stack(to_utm.transform(c[:, 0], c[:, 1])))

projected = project(source)
distance = report["distanceMetres"]
perimeter = projected.exterior.length
sides = perimeter * distance                      # the frame along the edges
corners = math.pi * distance ** 2                 # the four quarter circles, one full circle
added = report["fieldworkAreaSquareMetres"] - report["sourceAreaSquareMetres"]

print(f"perimeter of the study area   {perimeter:10.0f} m")
print(f"frame along the sides         {sides:10.0f} m2")
print(f"corners, as a full circle     {corners:10.0f} m2")
print(f"predicted addition            {sides + corners:10.0f} m2")
print(f"measured addition             {added:10.0f} m2")
print()
print(f"corners are {corners / (sides + corners):.1%} of what the buffer adds, and the polygon")
print(f"approximation touches only those -- so it costs about {corners * (1 - fraction(32)) / added:.3%} of the")
print("addition, which is why 8 segments is ample for this job.")

### Reading the result

`agrees` means the buffered area is within 0.1% of an independent buffer of the same boundary
by the same distance, and the result contains the original.

It does not mean 500 m is the right distance. The buffered boundary is an acquisition extent,
and how far past the study area it should reach is a judgement about the data being fetched —
a street network needs enough margin that routes near the edge are not truncated, which is
why this widget exists at all.